# DistilBert-CNN Intent Classifier — Kaggle GPU Training

**Self-contained notebook**: model definition + training pipeline.

**Instructions**:
1. Upload `Intent_Classifier_Model/data/` folder as a Kaggle dataset named `intent-classifier-data`
2. Enable **GPU T4** accelerator in notebook settings
3. Run All
4. Download `best_model.pt` + `training_results.json` from `/kaggle/working/`

Expected: ~2-3 min/epoch on T4 (vs 25 min on CPU)

In [ ]:
!pip install -q wandb

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
import pandas as pd
import numpy as np
from tqdm import tqdm
import time, json, math, re
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

start_time = time.time()

In [ ]:
# Update this to match your Kaggle dataset name
DATA_DIR = '/kaggle/input/intent-classifier-data'
OUTPUT_DIR = '/kaggle/working'

INTENT_NAMES = ['On-Topic Question', 'Off-Topic Question', 'Emotional-State', 'Pace-Related', 'Repeat/clarification', 'Debugging/Code-Sharing']
DEFAULT_CONFIDENCE_THRESHOLD = 0.65

# Verify data exists
for f in ['train.csv', 'val.csv', 'test.csv']:
    path = os.path.join(DATA_DIR, f)
    print(f"{'[OK]' if os.path.exists(path) else '[MISSING]'} {f}")

## Model Definition (DistilBert-CNN)

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_map = {
            'On-Topic Question': 0, 'Off-Topic Question': 1,
            'Emotional-State': 2, 'Pace-Related': 3, 'Repeat/clarification': 4,
            'Debugging/Code-Sharing': 5
        }
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        student_input = str(item.get('student_input', ''))
        session_context = str(item.get('session_context', ''))
        encoded = self.tokenizer(
            student_input, session_context,
            padding='max_length', truncation='longest_first',
            max_length=self.max_length, return_tensors='pt'
        )
        label_val = item.get('label', 0)
        if isinstance(label_val, str):
            label_val = self.label_map.get(label_val, 0)
        output = {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(label_val, dtype=torch.long)
        }
        if 'token_type_ids' in encoded:
            output['token_type_ids'] = encoded['token_type_ids'].squeeze(0)
        return output

class TinyBertCNN(nn.Module):
    def __init__(self, num_classes, bert_model_name='distilbert-base-uncased',
                 num_filters=192, filter_sizes=[3, 4, 5, 6], dropout=0.2,
                 hidden_dim=128, freeze_bert=False, temperature=1.0):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self._supports_token_type_ids = (
            hasattr(self.bert.config, 'type_vocab_size') and
            self.bert.config.type_vocab_size > 1
        )
        self.bert_hidden_size = self.bert.config.hidden_size
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
        self.convs = nn.ModuleList([
            nn.Conv1d(self.bert_hidden_size, num_filters, fs) for fs in filter_sizes
        ])
        self.batchnorms = nn.ModuleList([nn.GroupNorm(num_groups=32, num_channels=num_filters) for _ in filter_sizes])
        self.dropout = nn.Dropout(dropout)
        cnn_out_dim = len(filter_sizes) * num_filters + self.bert_hidden_size
        self.fc_hidden = nn.Linear(cnn_out_dim, hidden_dim)
        self.bn_hidden = nn.GroupNorm(num_groups=16, num_channels=hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.temperature = temperature

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        bert_kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None and self._supports_token_type_ids:
            bert_kwargs['token_type_ids'] = token_type_ids
        bert_output = self.bert(**bert_kwargs)
        sequence_output = bert_output.last_hidden_state
        cls_output = sequence_output[:, 0, :]
        sequence_output = sequence_output.transpose(1, 2)
        max_kernel = max(conv.kernel_size[0] for conv in self.convs)
        if sequence_output.size(2) < max_kernel:
            pad_size = max_kernel - sequence_output.size(2)
            sequence_output = nn.functional.pad(sequence_output, (0, pad_size))
        conv_outputs = []
        for conv, bn in zip(self.convs, self.batchnorms):
            conv_out = torch.relu(bn(conv(sequence_output)))
            pooled = torch.max_pool1d(conv_out, conv_out.size(2)).squeeze(2)
            conv_outputs.append(pooled)
        concatenated = torch.cat(conv_outputs + [cls_output], dim=1)
        concatenated = self.dropout(concatenated)
        hidden = torch.relu(self.bn_hidden(self.fc_hidden(concatenated)))
        logits = self.fc(hidden)
        return logits / self.temperature

class IntentClassifier:
    def __init__(self, num_classes, bert_model_name='distilbert-base-uncased',
                 num_filters=192, filter_sizes=[3, 4, 5, 6], dropout=0.2,
                 freeze_bert=False, device=None, temperature=1.0):
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = TinyBertCNN(
            num_classes=num_classes, bert_model_name=bert_model_name,
            num_filters=num_filters, filter_sizes=filter_sizes,
            dropout=dropout, freeze_bert=freeze_bert, temperature=temperature
        ).to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.num_classes = num_classes

    def train_step(self, batch, optimizer, criterion):
        self.model.train()
        input_ids = batch['input_ids'].to(self.device)
        attention_mask = batch['attention_mask'].to(self.device)
        labels = batch['labels'].to(self.device)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(self.device)
        logits = self.model(input_ids, attention_mask, token_type_ids=token_type_ids)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        optimizer.step()
        return loss.item()

    def train_step_amp(self, batch, optimizer, criterion, scaler=None):
        from torch.cuda.amp import autocast
        self.model.train()
        input_ids = batch['input_ids'].to(self.device)
        attention_mask = batch['attention_mask'].to(self.device)
        labels = batch['labels'].to(self.device)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(self.device)
        optimizer.zero_grad()
        with autocast():
            logits = self.model(input_ids, attention_mask, token_type_ids=token_type_ids)
            loss = criterion(logits, labels)
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            optimizer.step()
        return loss.item()

    def save_model(self, path, temperature=None):
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'num_classes': self.num_classes,
            'temperature': temperature or self.model.temperature,
        }
        torch.save(checkpoint, path)
        print(f'Model saved to {path}')

    def load_model(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.num_classes = checkpoint.get('num_classes', self.num_classes)
        if 'temperature' in checkpoint:
            self.model.temperature = checkpoint['temperature']
        print(f'Model loaded from {path}')

print('Model classes defined')

## Utilities

In [ ]:
def normalize_context(ctx):
    if not isinstance(ctx, str):
        return ''
    ctx = re.sub(r'\|\s*ability:[^|]+', '', ctx)
    ctx = re.sub(r'\s*\|\s*', ' | ', ctx).strip(' |')
    return ctx

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001, verbose=True, mode='min'):
        self.patience, self.min_delta, self.verbose, self.mode = patience, min_delta, verbose, mode
        self.counter, self.best_score, self.early_stop, self.best_epoch = 0, None, False, 0
    def __call__(self, score, epoch):
        improved = (
            self.best_score is None or
            (self.mode == 'min' and score < self.best_score - self.min_delta) or
            (self.mode == 'max' and score > self.best_score + self.min_delta)
        )
        if improved:
            self.best_score, self.best_epoch, self.counter = score, epoch, 0
        else:
            self.counter += 1
            if self.verbose: print(f'  Early stopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose: print(f'  [!] Early stopping triggered! Best epoch was {self.best_epoch}')

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps):
        self.optimizer, self.warmup_steps, self.total_steps = optimizer, warmup_steps, total_steps
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_step = 0
    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            scale = self.current_step / max(1, self.warmup_steps)
        else:
            progress = (self.current_step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            scale = 0.5 * (1.0 + math.cos(math.pi * progress))
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * scale

def compute_class_weights(labels, num_classes, device):
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    counts[counts == 0] = 1.0
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32).to(device)

def evaluate_model_full(classifier, loader):
    classifier.model.eval()
    all_preds, all_labels, total_loss = [], [], 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(classifier.device)
            attention_mask = batch['attention_mask'].to(classifier.device)
            labels = batch['labels'].to(classifier.device)
            token_type_ids = batch.get('token_type_ids')
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(classifier.device)
            logits = classifier.model(input_ids, attention_mask, token_type_ids=token_type_ids)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    n = len(all_labels)
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    return total_loss / n, accuracy, precision, recall, f1, all_preds, all_labels

print('Utilities defined')

## Load Data & Prepare

In [ ]:
BATCH_SIZE = 32
EPOCHS = 20
BERT_LR = 2e-5
HEAD_LR = 2e-4
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128
PATIENCE = 4
DROPOUT = 0.2
FREEZE_BERT = False

hyperparams = {
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'bert_lr': BERT_LR, 'head_lr': HEAD_LR,
    'weight_decay': WEIGHT_DECAY, 'max_length': MAX_LENGTH, 'patience': PATIENCE,
    'label_smoothing': 0.1, 'dropout': DROPOUT, 'freeze_bert': FREEZE_BERT
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

REAL_UTTERANCE_PATH = os.path.join(DATA_DIR, 'real_utterances.csv')
real_eval_df = None
if os.path.exists(REAL_UTTERANCE_PATH):
    real_df = pd.read_csv(REAL_UTTERANCE_PATH)
    real_df = real_df.sample(frac=1, random_state=42).reset_index(drop=True)
    split_idx = int(len(real_df) * 0.70)
    real_train_df = real_df.iloc[:split_idx]
    real_eval_df = real_df.iloc[split_idx:].copy()
    real_train_oversampled = pd.concat([real_train_df] * 2, ignore_index=True)
    train_df = pd.concat([train_df, real_train_oversampled], ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'[+] Mixed {len(real_train_oversampled)} real utterance rows. Train total: {len(train_df)}')

for df in [train_df, val_df, test_df]:
    df['session_context'] = df['session_context'].apply(normalize_context)
if real_eval_df is not None:
    real_eval_df['session_context'] = real_eval_df['session_context'].apply(normalize_context)

num_classes = train_df['label'].nunique()
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Classes: {num_classes}')

## Build Model & Train

In [ ]:
classifier = IntentClassifier(
    num_classes=num_classes, bert_model_name='distilbert-base-uncased',
    num_filters=192, filter_sizes=[3, 4, 5, 6],
    dropout=DROPOUT, freeze_bert=FREEZE_BERT, device=device
)

train_dataset = IntentDataset(train_df.to_dict('records'), classifier.tokenizer, max_length=MAX_LENGTH)
val_dataset = IntentDataset(val_df.to_dict('records'), classifier.tokenizer, max_length=MAX_LENGTH)
test_dataset = IntentDataset(test_df.to_dict('records'), classifier.tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_weights = compute_class_weights(train_df['label'].values, num_classes, classifier.device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1, weight=class_weights)

bert_params = list(classifier.model.bert.parameters())
head_params = [p for n, p in classifier.model.named_parameters() if not n.startswith('bert.')]
optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': BERT_LR},
    {'params': head_params, 'lr': HEAD_LR}
], weight_decay=WEIGHT_DECAY)

total_steps = len(train_loader) * EPOCHS
scheduler = WarmupCosineScheduler(optimizer, int(total_steps * 0.1), total_steps)
early_stopping = EarlyStopping(patience=PATIENCE, mode='max')
best_val_f1 = 0.0
best_model_path = os.path.join(OUTPUT_DIR, 'best_model.pt')
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

os.environ['WANDB_MODE'] = 'online'
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret('WANDB_API_KEY')
wandb.login(key=wandb_api_key)

wandb.init(project='intent-classifier',
           name=f'kaggle-{int(time.time())}',
           config={**hyperparams, 'train_rows': len(train_df), 'device': str(device),
                   'filter_sizes': [3, 4, 5, 6], 'num_filters': 192, 'norm_type': 'GroupNorm'})
print(f'Ready: {total_steps} steps, batch_size={BATCH_SIZE}')

In [ ]:
from torch.cuda.amp import GradScaler

class _LoopState:
    restart_count: int = 0
_state = _LoopState()

def fit_temperature(clf, v_loader, max_iter=50):
    '''Fit temperature parameter T to minimize NLL on validation set.'''
    clf.model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in v_loader:
            input_ids = batch['input_ids'].to(clf.device)
            attention_mask = batch['attention_mask'].to(clf.device)
            labels_b = batch['labels'].to(clf.device)
            token_type_ids = batch.get('token_type_ids')
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(clf.device)
            logits = clf.model(input_ids, attention_mask, token_type_ids=token_type_ids)
            logits = logits * clf.model.temperature
            all_logits.append(logits)
            all_labels.append(labels_b)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    
    best_T, best_nll = 1.0, float('inf')
    for T in [0.5, 0.7, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, 5.0]:
        scaled = all_logits / T
        nll = nn.CrossEntropyLoss()(scaled, all_labels).item()
        if nll < best_nll:
            best_nll, best_T = nll, T
    
    for T in [best_T - 0.3, best_T - 0.15, best_T, best_T + 0.15, best_T + 0.3]:
        if T <= 0: continue
        scaled = all_logits / T
        nll = nn.CrossEntropyLoss()(scaled, all_labels).item()
        if nll < best_nll:
            best_nll, best_T = nll, T
    
    clf.model.temperature = best_T
    print(f'[+] Temperature fitted: T={best_T:.3f} (val NLL={best_nll:.4f})')
    return best_T

scaler = GradScaler()

for epoch in range(EPOCHS):
    classifier.model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch in pbar:
        loss = classifier.train_step_amp(batch, optimizer, criterion, scaler=scaler)
        scheduler.step()
        train_loss += loss
        pbar.set_postfix({'loss': f'{loss:.4f}'})

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc, _, _, val_f1, preds, labels = evaluate_model_full(classifier, val_loader)

    history['train_loss'].append(round(avg_train_loss, 4))
    history['val_loss'].append(round(val_loss, 4))
    history['val_acc'].append(round(val_acc, 4))
    history['val_f1'].append(round(val_f1, 4))

    print(f'Epoch {epoch+1}: Train={avg_train_loss:.4f} Val_Loss={val_loss:.4f} Val_Acc={val_acc:.4f} Val_F1={val_f1:.4f}')

    _, _, pcf1, _ = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    log = {'epoch': epoch+1, 'train_loss': avg_train_loss, 'val_loss': val_loss, 'val_f1': val_f1}
    for i, n in enumerate(INTENT_NAMES):
        if i < len(pcf1): log[f'f1_{n.replace(" ","_").replace("/","_")}'] = float(pcf1[i])
    wandb.log(log)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        classifier.save_model(best_model_path)
        print(f'  [+] Best F1: {val_f1:.4f}')

    early_stopping(val_f1, epoch + 1)
    if early_stopping.early_stop:
        MAX_RESTARTS = 2

        if _state.restart_count >= MAX_RESTARTS:
            print(f"Early stopping — {MAX_RESTARTS} restarts exhausted. Terminating at epoch {epoch + 1}.")
            break

        best_epoch = early_stopping.best_epoch
        print(f"Early stopping at epoch {epoch + 1} — rewinding to epoch {best_epoch} (best val_f1={best_val_f1:.4f})")

        classifier.load_model(best_model_path)

        temperature = fit_temperature(classifier, val_loader)
        classifier.model.temperature = temperature
        print(f"Temperature recalibrated: T={temperature:.4f}")

        for pg in optimizer.param_groups:
            pg['lr'] = pg['lr'] * 0.5
        print(f"LRs halved — BERT: {optimizer.param_groups[0]['lr']:.2e}  HEAD: {optimizer.param_groups[-1]['lr']:.2e}")

        if hasattr(criterion, 'label_smoothing'):
            criterion.label_smoothing = max(criterion.label_smoothing * 0.5, 0.01)
            print(f"Label smoothing -> {criterion.label_smoothing:.3f}")

        remaining = EPOCHS - (epoch + 1)
        warmup    = min(200, remaining * len(train_loader) // 10)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup,
            num_training_steps=remaining * len(train_loader),
        )
        print(f"Scheduler reset — {warmup} warmup / {remaining * len(train_loader)} total steps")

        early_stopping.counter    = 0
        early_stopping.early_stop = False

        _state.restart_count += 1
        print(f"Restart {_state.restart_count} / {MAX_RESTARTS} — continuing training.")

print(f'\nTraining done. Best val F1: {best_val_f1:.4f}')

## Evaluation & Save

In [ ]:
classifier.load_model(best_model_path)

# Temperature already fitted during final early stopping or earlier, but we can do one last fit
best_T = fit_temperature(classifier, val_loader)

test_loss, test_acc, test_prec, test_rec, test_f1, preds, labels = evaluate_model_full(classifier, test_loader)
duration = round(time.time() - start_time, 2)

pp, pr, pf, ps = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
pcm = {}
for i, n in enumerate(INTENT_NAMES):
    pcm[n] = {'precision': round(float(pp[i]),4), 'recall': round(float(pr[i]),4),
              'f1': round(float(pf[i]),4), 'support': int(ps[i])}

cm = confusion_matrix(labels, preds).tolist()
report = classification_report(labels, preds, target_names=INTENT_NAMES, zero_division=0)

print(f'Test Acc: {test_acc:.4f} | F1: {test_f1:.4f} | Loss: {test_loss:.4f} | Time: {duration}s')
print(report)
print('Confusion Matrix:')
for row in cm: print(f'  {row}')

# Real utterance eval
if real_eval_df is not None and len(real_eval_df) > 0:
    print('\n' + '='*60 + '\nREAL-UTTERANCE EVALUATION\n' + '='*60)
    rd = IntentDataset(real_eval_df.to_dict('records'), classifier.tokenizer, max_length=MAX_LENGTH)
    rl = DataLoader(rd, batch_size=BATCH_SIZE, shuffle=False)
    rloss, racc, rp, rr, rf1, rpreds, rlabels = evaluate_model_full(classifier, rl)
    print(f'Real Acc: {racc:.4f} | F1: {rf1:.4f}')
    print(classification_report(rlabels, rpreds, target_names=INTENT_NAMES, zero_division=0))

results = {
    'model': 'DistilBert-CNN', 'duration_s': duration, 'epochs': len(history['train_loss']),
    'metrics': {'acc': round(test_acc,4), 'f1': round(test_f1,4), 'loss': round(test_loss,4)},
    'per_class': pcm, 'confusion_matrix': cm, 'history': history, 'report': report,
    'temperature': best_T
}
with open(os.path.join(OUTPUT_DIR, 'training_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

wandb.log({'test_f1': test_f1, 'test_acc': test_acc, 'temperature': best_T})
wandb.finish()

print(f'\nDownload best_model.pt and training_results.json from /kaggle/working/')